# El Motor de Recomendación de "GameStore" 🎮
Contexto del Negocio:
Han sido contratados por "GameStore", una tienda local de videojuegos. El gerente quiere rediseñar su sitio web e implementar la función: "Los usuarios que compraron esto, también llevaron...".

Su misión es analizar las transacciones del fin de semana, aplicar el algoritmo de Reglas de Asociación y decidir qué productos deben recomendarse activamente en la pantalla de pago (Checkout).

## Fase 1: Los Tickets de Compra
A continuación, se presenta la lista de compras del fin de semana. Nuestro catálogo principal incluye Consolas, Controles, Juegos y Suscripciones.

In [1]:
%pip install mlxtend

INFO: pip is looking at multiple versions of mlxtend to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 20.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 23.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [mlxtend]m4/6 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install "numpy<2" "mlxtend==0.23.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 17.1 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Importamos las librerías necesarias
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. DATOS DE PRODUCCIÓN (Listas de compras)
tickets_gaming = [
    ['Consola PS5', 'Control Extra', 'FIFA 26'],
    ['Consola PS5', 'Suscripción Plus', 'Cable USB-C'],
    ['Control Extra', 'Cable USB-C'],
    ['Consola PS5', 'Control Extra', 'Suscripción Plus'],
    ['Control Extra', 'FIFA 26'],
    ['Consola PS5', 'Control Extra', 'Cable USB-C'],
    ['Cable USB-C', 'Suscripción Plus'],
    ['Control Extra'],
    ['Consola PS5', 'Control Extra', 'FIFA 26', 'Suscripción Plus'],
    ['Control Extra', 'Funda de Silicon']
]

# --- RETO 1: TRANSFORMACIÓN (Escribe tu código abajo) ---
# Convierte la lista 'tickets_gaming' en un DataFrame booleano llamado 'df_tienda'
# usando TransactionEncoder.

te = TransactionEncoder()
matriz = te.fit(tickets_gaming).transform(tickets_gaming)
df_tienda = pd.DataFrame(matriz, columns=te.columns_)

print("--- MATRIZ TRANSACCIONAL ---")
display(df_tienda)

--- MATRIZ TRANSACCIONAL ---


,Cable USB-C,Consola PS5,Control Extra,FIFA 26,Funda de Silicon,Suscripción Plus
0,False,True,True,True,False,False
1,True,True,False,False,False,True
2,True,False,True,False,False,False
3,False,True,True,False,False,True
4,False,False,True,True,False,False
5,True,True,True,False,False,False
6,True,False,False,False,False,True
7,False,False,True,False,False,False
8,False,True,True,True,False,True
9,False,False,True,False,True,False


### Fase 2: Extracción de Reglas y Filtrado Direccional
Vamos a generar todas las reglas posibles con un Soporte mínimo del 20% (0.2) y una Confianza mínima del 40% (0.4).

In [4]:
# --- RETO 2: EXTRACCIÓN (Escribe tu código abajo) ---

itemsets = apriori(df_tienda, min_support=0.2, use_colnames=True)
reglas_store = association_rules(itemsets, metric='confidence', min_threshold=0.4)

# Creamos columnas auxiliares de texto para facilitar el filtrado de Pandas
reglas_store['ant_str'] = reglas_store['antecedents'].apply(lambda x: ', '.join(list(x)))
reglas_store['con_str'] = reglas_store['consequents'].apply(lambda x: ', '.join(list(x)))

print("--- TOP 5 REGLAS POR CONFIANZA ---")
# Muestra las 5 reglas con mayor confianza
# display(reglas_store.sort_values(by='confidence', ascending=False).head())

--- TOP 5 REGLAS POR CONFIANZA ---


### Fase 3: La Batalla Direccional (PS5 vs Control Extra)

El gerente tiene una duda específica: *"Si alguien mete una Consola PS5 al carrito, ¿debo recomendarle un Control Extra? ¿Y si alguien mete un Control Extra, le recomiendo la Consola PS5?"*

Filtra tu tabla `reglas_store` para aislar exclusivamente la interacción entre estos dos productos y comparar sus métricas.

In [5]:
# --- RETO 3: EL FILTRO (Escribe tu código abajo) ---
dilema_ps5 = reglas_store[
    ((reglas_store['ant_str'] == 'Consola PS5') & (reglas_store['con_str'] == 'Control Extra')) |
    ((reglas_store['ant_str'] == 'Control Extra') & (reglas_store['con_str'] == 'Consola PS5'))
]

print("--- ANÁLISIS DIRECCIONAL (Consola vs Control) ---")
display(dilema_ps5[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

--- ANÁLISIS DIRECCIONAL (Consola vs Control) ---


,antecedents,consequents,support,confidence,lift
5,(Control Extra),(Consola PS5),0.4,0.5,1.0
6,(Consola PS5),(Control Extra),0.4,0.8,1.0


### 🧠 Fase 4: Toma de Decisiones (Responde aquí mismo)

Revisa el resultado del Reto 3 y responde al gerente con argumentos basados en datos:

**1. Interpretación de Confianza Asimétrica:**
Observa la diferencia de Confianza entre comprar la Consola $\rightarrow$ Control, frente a comprar el Control $\rightarrow$ Consola. Explica con lógica de negocio o comportamiento del consumidor por qué ocurre esta asimetría tan grande en los porcentajes.
> **Tu Respuesta:** La confianza se calcula dividiendo entre el soporte del antecedente, y los dos productos tienen popularidad muy distinta. El Control Extra está en el 80% de los tickets y la PS5 en el 50%. Los dos comparten el mismo soporte conjunto (0.4), pero PS5 → Control da 0.4/0.5 = 80% y Control → PS5 da 0.4/0.8 = 50%.
En términos de negocio, quien compra una consola casi siempre se lleva otro control para jugar con alguien. En cambio, mucha gente compra el control solo, porque ya tiene consola (o porque se le rompió el suyo), así que no tiene sentido que también compre una PS5.
Lo importante es que el lift = 1.0 en ambas direcciones. Ese 80% no indica una relación especial: es exactamente lo que esperaríamos por azar, porque 8 de cada 10 clientes compran control de todas formas.

**2. Diseño del Sistema de Recomendación:**
Basado en los datos, si un cliente está a punto de pagar únicamente un **Control Extra** en su carrito de compras, ¿sería una buena idea que el sistema lance un *Pop-up* recomendándole comprar una Consola PS5? Justifica usando las métricas descubiertas.
> **Tu Respuesta:** No. La confianza es solo de 50% y el lift es 1.0, así que comprar un control no aumenta para nada la probabilidad de comprar una consola. Además, el leverage es 0 (no hay ventas extra) y la convicción es 1 (independencia). Lo más probable es que ese cliente ya tenga consola. Recomendarle una PS5 de $10k sería irrelevante y molesto. Le serviría más algo con lift > 1, como FIFA 26 o la Suscripción Plus.

### Fase 5: Evaluando el Impacto en Volumen (Leverage)

El *Lift* nos dice qué tan fuerte es la asociación, pero es una proporción. No nos dice cuántas ventas adicionales reales estamos logrando. Para eso existe el **Leverage** (Apalancamiento).

**La Matemática:** $Leverage(A \rightarrow B) = Soporte(A \cap B) - [Soporte(A) \times Soporte(B)]$

El *Leverage* mide la diferencia entre observar A y B juntos en la vida real, frente a lo que esperaríamos si se vendieran de forma totalmente aleatoria. Un valor de 0 indica independencia. Un valor positivo mayor indica más ventas conjuntas "extra".

### Fase 6: Nivel de Dependencia (Conviction)

Por último, el gerente quiere saber si hay accesorios que sean "esclavos" de la venta principal. Para esto usamos la **Convicción** (*Conviction*).

**La Matemática:** $Conviction(A \rightarrow B) = \frac{1 - Soporte(B)}{1 - Confianza(A \rightarrow B)}$

La Convicción evalúa qué tan dependiente es el Consecuente respecto al Antecedente. Un valor alto de convicción significa que es altísimamente improbable que el Antecedente ocurra sin el Consecuente. Si la Confianza es del 100% (1.0), la Convicción será infinita (`inf`).

In [6]:
# --- RETO 5: NIVEL DE DEPENDENCIA (Escribe tu código abajo) ---
# 1. Ordena tu DataFrame 'reglas_store' por la columna 'conviction' de forma descendente.
# 2. Muestra las 3 mejores reglas.

print("--- TOP 3 REGLAS POR CONVICCIÓN (Dependencia del producto) ---")
top_conviction = reglas_store.sort_values(by='conviction', ascending=False)
display(top_conviction[['antecedents', 'consequents', 'confidence', 'lift', 'leverage', 'conviction']].head(3))

--- TOP 3 REGLAS POR CONVICCIÓN (Dependencia del producto) ---


,antecedents,consequents,confidence,lift,leverage,conviction
11,(FIFA 26),(Control Extra),1.0,1.25,0.06,inf
18,"(Control Extra, Suscripción Plus)",(Consola PS5),1.0,2.00,0.10,inf
15,"(FIFA 26, Consola PS5)",(Control Extra),1.0,1.25,0.04,inf


### 🧠 Fase 7: El Diagnóstico Final (Responde aquí)

Has utilizado métricas avanzadas que la mayoría de los analistas junior ignoran. Responde las últimas preguntas del gerente de "GameStore":

**1. El Significado del Leverage Positivo:**
En contraste con el Lift (que puede ser un número enorme pero representar solo 1 venta rara), el Leverage se enfoca en el volumen general de la tienda. Si una regla tiene un Leverage de `0.12`, ¿qué le dirías al gerente sobre el impacto de esa combinación en su inventario comparado con si esos productos se vendieran al azar?
> **Tu Respuesta:** Significa que esa combinación aparece en un 12% de todas las transacciones de la tienda por encima de lo que pasaría si los productos se vendieran al azar. Por cada 100 tickets hay unas 12 ventas conjuntas “extra” que se explican por la relación entre los productos y no por coincidencia. A diferencia del lift, esto mide volumen real: dice cuánto inventario se mueve gracias a la combinación. Como referencia, en nuestros datos el leverage positivo más alto es 0.10, y aparece un -0.12 en Cable USB-C → Control (ventas que se pierden comparado con el azar).

**2. El Misterio del Infinito:**
Si en tu tabla de Convicción aparece el valor `inf` (infinito) para la regla `(FIFA 26) -> (Control Extra)`, ¿qué significa eso exactamente en el comportamiento de los clientes que entraron a comprar el juego de fútbol este fin de semana? 
> **Tu Respuesta:** La confianza es 100%: los 3 clientes que compraron FIFA 26 se llevaron también un Control Extra. Con la fórmula, (1 − 0.8) / (1 − 1.0) = 0.2 / 0 = ∞. Eso quiere decir que este fin de semana nadie compró FIFA sin control, porque el juego se disfruta en multijugador local y sin un segundo control no sirve igual. Pero hay que tomarlo con cuidado: son solo 3 tickets, y un solo cliente que compre FIFA sin control haría que la convicción bajara a un número finito.

### Fase 8: De Asociación a Correlación Estadística (El Engaño de la Confianza)

A veces, una regla tiene alta **Confianza** simplemente porque el producto consecuente es extremadamente popular por sí solo, ocultando el hecho de que ambos productos en realidad se **repelen** (correlación negativa).

Para auditar nuestras reglas con rigor estadístico, usaremos dos técnicas de **Análisis de Correlación**:
1. **La Métrica de Zhang (`zhangs_metric`):** Mide la correlación real entre -1 (repulsión total) y +1 (atracción total), corrigiendo el sesgo de los productos hiper-populares.
2. **La Matriz de Correlación Phi ($\phi$):** El equivalente a la correlación de Pearson pero para matrices booleanas (True/False).

In [7]:
# --- DEMOSTRACIÓN DE LA PARADOJA (Ejecutar y analizar) ---
# Filtramos qué pasa cuando alguien compra 'Funda de Silicon' o 'FIFA 26' hacia los productos populares
paradoja = reglas_store[
    (reglas_store['ant_str'] == 'Cable USB-C') &
    (reglas_store['con_str'] == 'Control Extra')
]

print("--- ¿IMPULSO REAL O CORRELACIÓN NEGATIVA? ---")
# Observa cómo incluimos 'zhangs_metric' que mlxtend calcula automáticamente
display(paradoja[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'zhangs_metric']])

--- ¿IMPULSO REAL O CORRELACIÓN NEGATIVA? ---


,antecedents,consequents,support,confidence,lift,zhangs_metric
2,(Cable USB-C),(Control Extra),0.2,0.5,0.625,-0.5


### Reto 6: La Matriz de Correlación Phi ($\phi$) de la Tienda

Antes de lanzar promociones, el director de datos quiere ver un panorama general de qué productos "se llevan bien" (correlación positiva) y qué productos "compiten o se estorban" (correlación negativa) en toda la tienda.

**Tu tarea:**
1. Calcula la matriz de correlación directamente sobre tu DataFrame booleano `df_tienda` usando el método `.corr()`.
2. Extrae y ordena las correlaciones específicas de la **Consola PS5** contra el resto del catálogo.

In [8]:
# --- RETO 6: MATRIZ DE CORRELACIÓN (Escribe tu código abajo) ---

# 1. Calcula la matriz de correlación de df_tienda
matriz_corr = df_tienda.corr()

print("--- MATRIZ DE CORRELACIÓN PHI (TODA LA TIENDA) ---")
display(matriz_corr.round(2))

print("\n--- CORRELACIÓN CON CONSOLA PS5 ---")
corr_ps5 = matriz_corr['Consola PS5'].drop('Consola PS5').sort_values(ascending=False)
display(corr_ps5)

--- MATRIZ DE CORRELACIÓN PHI (TODA LA TIENDA) ---


,Cable USB-C,Consola PS5,Control Extra,FIFA 26,Funda de Silicon,Suscripción Plus
Cable USB-C,1.00,0.00,-0.61,-0.53,-0.27,0.17
Consola PS5,0.00,1.00,-0.00,0.22,-0.33,0.41
Control Extra,-0.61,-0.00,1.00,0.33,0.17,-0.61
FIFA 26,-0.53,0.22,0.33,1.00,-0.22,-0.09
Funda de Silicon,-0.27,-0.33,0.17,-0.22,1.00,-0.27
Suscripción Plus,0.17,0.41,-0.61,-0.09,-0.27,1.00



--- CORRELACIÓN CON CONSOLA PS5 ---


Suscripción Plus    4.082483e-01
FIFA 26             2.182179e-01
Cable USB-C         0.000000e+00
Control Extra      -2.097088e-17
Funda de Silicon   -3.333333e-01
Name: Consola PS5, dtype: float64

### 🧠 Fase 9: Defensa de Correlación (Responde aquí)

Analiza tu Matriz de Correlación Phi ($\phi$) y las métricas de Zhang para responder las siguientes preguntas de auditoría:

**1. Productos Sustitutos / Enemigos (Correlación Negativa):**
Observa la correlación entre `Control Extra` y `Cable USB-C` (o entre `Consola PS5` y `Funda de Silicon`). Notarás que tienen valores negativos (ej. `-0.35` o `-0.50`). 
Sin embargo, la regla `Cable USB-C -> Control Extra` apareció en nuestro reporte original con una **Confianza del 50%**. 
¿Por qué es un grave error financiero armar un combo con estos dos productos a pesar de tener 50% de Confianza? Explica usando el concepto de Correlación.
> **Tu Respuesta:** El 50% de confianza engaña porque el Control Extra ya está en el 80% de todos los tickets. Comprar un cable no aumenta la probabilidad de llevarse un control; la baja de 80% a 50%. Todas las métricas apuntan a lo mismo: lift 0.625 (< 1), Zhang -0.50 y correlación phi -0.61, una de las más negativas de la tienda.
Son productos que se repelen. Probablemente el que compra cable ya tiene control y solo necesita cargarlo, o sea, son casi sustitutos. Armar un combo significaría darle descuento al control a gente que de todas formas lo iba a comprar a precio completo (80% de base), y no generaría ventas nuevas. Se pierde margen sin ganar volumen.

**2. Ventaja Operativa en Ingeniería:**
Calcular el algoritmo Apriori en una tienda con 50,000 productos puede colapsar la memoria RAM por la explosión combinatoria. En cambio, calcular `df.corr()` es una operación matricial directa y muy rápida en C++/CUDA. ¿Cómo usarías la matriz de correlación como filtro previo antes de correr Apriori?
> **Tu Respuesta:** Primero quito los productos con soporte individual menor al mínimo (df.mean()), que es casi gratis y elimina la mayor parte del catálogo.
Con los que quedan calculo la correlación phi. Con una matriz booleana dispersa es una multiplicación Xᵀ·X que corre en GPU.
Me quedo solo con los productos que tengan al menos una correlación > umbral (ej. 0.1) con otro, y los agrupo en clústeres de productos que “se llevan bien”.
Corro Apriori por clúster, con decenas de columnas en lugar de 50,000. Así la explosión combinatoria se controla y no se generan reglas entre productos con correlación negativa, que de todos modos tiraríamos.